# UPI Transaction Fraud Detection
## Exploratory Data Analysis & Modelling Walkthrough
**Author:** Gayam Sai Kowshik Reddy | NIT Raipur

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, re

from data_generator import generate_transactions
from preprocessing import compute_rfm, engineer_features, build_features
from train import MODELS, evaluate

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

## 1. Dataset Generation

In [ ]:
# Load or generate
CSV_PATH = '../data/upi_transactions.csv'
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f'Loaded from disk: {len(df):,} rows')
else:
    df = generate_transactions()
    os.makedirs('../data', exist_ok=True)
    df.to_csv(CSV_PATH, index=False)
    print(f'Generated: {len(df):,} rows')

print(f"Fraud rate: {df.is_fraud.mean()*100:.2f}%")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 2a. Class imbalance
ax = axes[0, 0]
counts = df.is_fraud.value_counts()
ax.bar(['Legit', 'Fraud'], counts, color=['#2ecc71', '#e74c3c'], edgecolor='white')
ax.set_title('Class Distribution', fontweight='bold')
ax.set_ylabel('Count')
for i, v in enumerate(counts):
    ax.text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# 2b. Amount distribution by label
ax = axes[0, 1]
for label, color, name in [(0,'#2ecc71','Legit'), (1,'#e74c3c','Fraud')]:
    data = np.log1p(df[df.is_fraud==label]['amount'])
    ax.hist(data, bins=50, alpha=0.6, color=color, label=name)
ax.set_title('Log(Amount) Distribution', fontweight='bold')
ax.set_xlabel('log(1 + amount)')
ax.legend()

# 2c. Fraud by hour
ax = axes[1, 0]
hourly = df.groupby('hour_of_day')['is_fraud'].mean() * 100
ax.bar(hourly.index, hourly.values, color='#e67e22')
ax.axhline(df.is_fraud.mean()*100, color='red', linestyle='--', label='Overall avg')
ax.set_title('Fraud Rate by Hour of Day', fontweight='bold')
ax.set_xlabel('Hour'); ax.set_ylabel('Fraud Rate (%)')
ax.legend()

# 2d. Fraud by category
ax = axes[1, 1]
cat_fraud = df.groupby('category')['is_fraud'].mean().sort_values(ascending=True) * 100
ax.barh(cat_fraud.index, cat_fraud.values, color='#9b59b6')
ax.set_title('Fraud Rate by Category', fontweight='bold')
ax.set_xlabel('Fraud Rate (%)')

plt.suptitle('UPI Fraud EDA — Key Patterns', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/eda_overview.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (numeric features)
num_cols = ['amount', 'hour_of_day', 'velocity_1hr', 'failure_count_24hr',
            'new_device', 'location_mismatch', 'prior_fraud_flag', 'is_fraud']
plt.figure(figsize=(10, 8))
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/correlation_matrix.png', bbox_inches='tight')
plt.show()

## 3. RFM Segmentation

In [ ]:
rfm = compute_rfm(df)
print(rfm.describe())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, color in zip(axes, ['recency', 'frequency', 'monetary'],
                           ['#3498db', '#2ecc71', '#e74c3c']):
    rfm[col].hist(ax=ax, bins=40, color=color, edgecolor='white')
    ax.set_title(f'RFM: {col.capitalize()}', fontweight='bold')
plt.suptitle('RFM Distribution per Sender', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Model Training & Evaluation

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

X_train, X_test, y_train, y_test, scaler, feature_cols = build_features(df)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
all_results = []
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for (name, model), ax in zip(MODELS.items(), axes):
    print(f'\nTraining {name}...')
    model.fit(X_train, y_train)
    metrics = evaluate(name, model, X_test, y_test)
    all_results.append(metrics)
    
    # Confusion matrix plot
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test,
        display_labels=['Legit', 'Fraud'],
        cmap='Blues', ax=ax
    )
    ax.set_title(f'{name}\nF1={metrics["f1_score"]:.3f}', fontweight='bold')

plt.suptitle('Confusion Matrices — All Models', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# Model comparison bar chart
metrics_df = pd.DataFrame(all_results)[['model','accuracy','precision','recall','f1_score','roc_auc']]
metrics_df = metrics_df.set_index('model')

ax = metrics_df.plot(kind='bar', figsize=(12, 6), edgecolor='white',
                     color=['#3498db','#2ecc71','#e74c3c','#9b59b6','#f39c12'])
ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=14)
ax.set_ylabel('Score')
ax.set_ylim(0.7, 1.0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../reports/model_comparison.png', bbox_inches='tight')
plt.show()

print('\nFinal Metrics:')
display(metrics_df.round(4))

## 5. Feature Importance (Random Forest)

In [ ]:
from train import log_feature_importance
rf_model = MODELS['Random Forest']
feat_imp = log_feature_importance(rf_model, feature_cols, top_n=15)

# Plot
imp_df = pd.DataFrame(feat_imp).sort_values('importance')
plt.figure(figsize=(10, 7))
plt.barh(imp_df.feature, imp_df.importance, color='#e67e22')
plt.xlabel('Feature Importance')
plt.title('Top 15 Features — Random Forest', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/feature_importance.png', bbox_inches='tight')
plt.show()

## 6. Inference Demo

In [ ]:
# Score a suspicious transaction
test_txns = [
    {"transaction_id": "TEST001", "amount": 87500, "hour_of_day": 2,
     "new_device": 1, "location_mismatch": 1, "velocity_1hr": 18,
     "prior_fraud_flag": 0, "failure_count_24hr": 4,
     "sender_upi": "usr12345x@okaxis", "receiver_upi": "merchant@oksbi",
     "is_weekend": 0, "sender_upi_valid": 1, "receiver_upi_valid": 1},
    {"transaction_id": "TEST002", "amount": 350, "hour_of_day": 14,
     "new_device": 0, "location_mismatch": 0, "velocity_1hr": 2,
     "prior_fraud_flag": 0, "failure_count_24hr": 0,
     "sender_upi": "priya42@oksbi", "receiver_upi": "zomato@paytm",
     "is_weekend": 1, "sender_upi_valid": 1, "receiver_upi_valid": 1}
]

from predict import score_transactions
bundle = {'model': rf_model, 'scaler': scaler,
          'feature_cols': feature_cols, 'model_name': 'Random Forest'}
results = score_transactions(test_txns, bundle)

for r in results:
    print(f"\n[{r['transaction_id']}]")
    print(f"  Fraud Probability : {r['fraud_probability']:.4f}")
    print(f"  Risk Label        : {r['risk_label']}")
    print(f"  Explanation       : {r['explanation']}")